# Interactive Subsidence Calculator
## Jupyter Notebook with Plotly Visualizations & Flask Web Server

This notebook provides an interactive interface for subsidence calculations with real-time visualization and a Flask backend server for web-based access.

---

## Section 1: Set Up Python Environment and Install Dependencies

In [ ]:
import sys
import subprocess

# Install required packages
packages = ['plotly', 'flask', 'numpy', 'pandas', 'ipywidgets']

for package in packages:
    try:
        __import__(package)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✓ {package} installed successfully")

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
import os

# Add the workspace directory to path
sys.path.insert(0, '/workspaces/Subsidence')

# Import calculator modules
from subsidence_calculator import create_report

print("✓ All imports successful")

---

## Section 2: Interactive Parameter Controls

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Create input widgets
title_label = widgets.HTML("<h3>Panel Geometry</h3>")
num_points_widget = widgets.IntSlider(value=4, min=3, max=10, description='Points:', style={'description_width': '100px'})

# Panel points input - Create 4 point inputs by default
point_inputs = []
for i in range(4):
    e_widget = widgets.FloatText(value=float([0, 100, 100, 0][i]), description=f'P{i+1} E:', style={'description_width': '80px'})
    n_widget = widgets.FloatText(value=float([0, 0, 100, 100][i]), description=f'P{i+1} N:', style={'description_width': '80px'})
    point_inputs.append((e_widget, n_widget))

# Subsidence parameters
params_label = widgets.HTML("<h3>Subsidence Parameters</h3>")
thickness_widget = widgets.FloatSlider(value=2.5, min=0.1, max=10, step=0.1, description='h (m):', style={'description_width': '100px'})
depth_widget = widgets.FloatSlider(value=300, min=50, max=1000, step=10, description='H (m):', style={'description_width': '100px'})
extraction_widget = widgets.RadioButtons(options={'Longwall (1.0)': 1.0, 'Board & Pillar (0.7)': 0.7}, description='Extraction:', style={'description_width': '100px'})
subsidence_factor_widget = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.1, description='a:', style={'description_width': '100px'})
new_ratio_widget = widgets.FloatSlider(value=1.4, min=0.5, max=3.0, step=0.1, description='W(new)/H:', style={'description_width': '100px'})
angle_widget = widgets.FloatSlider(value=35, min=0, max=90, step=1, description='α (deg):', style={'description_width': '100px'})
mesh_widget = widgets.FloatSlider(value=25, min=1, max=100, step=1, description='Mesh (m):', style={'description_width': '100px'})

# Calculate button
calc_button = widgets.Button(description='Calculate & Plot', button_style='info', tooltip='Calculate subsidence and generate visualization')

# Output area for visualization
output_area = widgets.Output()

print("✓ Interactive widgets created")

In [ ]:
def create_visualization(report):
    """Create interactive Plotly visualization from report"""
    points = report["points"]
    if not points:
        return None

    e = [p["easting"] for p in points]
    n = [p["northing"] for p in points]
    s = [p["subsidence"] for p in points]

    # Main scatter plot
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=e, y=n,
        mode='markers',
        marker=dict(
            size=8,
            color=[val * 1000 for val in s],
            colorscale='RdYlGn_r',
            showscale=True,
            colorbar=dict(title='Subsidence<br>(mm)'),
            line=dict(width=0.5, color='white'),
        ),
        text=[f'E: {ei:.2f}<br>N: {ni:.2f}<br>S: {si:.4f} m' for ei, ni, si in zip(e, n, s)],
        hovertemplate='%{text}<extra></extra>',
        name='Grid Points'
    ))

    # Panel boundary
    panel_points = report["inputs"]["panel_points"]
    if panel_points:
        px = [p["easting"] for p in panel_points] + [panel_points[0]["easting"]]
        py = [p["northing"] for p in panel_points] + [panel_points[0]["northing"]]
        fig.add_trace(go.Scatter(
            x=px, y=py,
            mode='lines+markers',
            name='Panel Boundary',
            line=dict(color='navy', width=3),
            marker=dict(size=8, color='navy'),
            hoverinfo='skip',
        ))

    # Centroid
    centroid = report["centroid"]
    fig.add_trace(go.Scatter(
        x=[centroid["easting"]], y=[centroid["northing"]],
        mode='markers',
        marker=dict(size=14, color='red', symbol='star'),
        name='Centroid',
        hovertemplate='Centroid<extra></extra>',
    ))

    # Influence circles
    bounds = report["bounds"]
    influence_dist = bounds["influence_distance"]
    for i, p in enumerate(panel_points, 1):
        theta = np.linspace(0, 2*np.pi, 72)
        circle_x = p["easting"] + influence_dist * np.cos(theta)
        circle_y = p["northing"] + influence_dist * np.sin(theta)
        fig.add_trace(go.Scatter(
            x=circle_x, y=circle_y,
            mode='lines',
            name=f'Influence {i}',
            line=dict(color='crimson', width=1, dash='dot'),
            hoverinfo='skip',
        ))

    fig.update_layout(
        title=f'Subsidence Map (s_max = {report["calculations"]["s_max"]:.2f} m)',
        xaxis_title='Easting (m)',
        yaxis_title='Northing (m)',
        yaxis=dict(scaleanchor='x', scaleratio=1),
        template='plotly_white',
        hovermode='closest',
        height=700,
    )
    
    return fig

print("✓ Visualization function created")

In [ ]:
def on_calculate(b):
    """Button click handler for calculation"""
    with output_area:
        clear_output(wait=True)
        
        try:
            # Collect panel points
            panel_points = []
            num_points = num_points_widget.value
            for i in range(num_points):
                if i < len(point_inputs):
                    panel_points.append({
                        "easting": point_inputs[i][0].value,
                        "northing": point_inputs[i][1].value
                    })
            
            # Create report
            report = create_report(
                panel_points,
                thickness_widget.value,
                depth_widget.value,
                extraction_widget.value,
                subsidence_factor_widget.value,
                new_ratio_widget.value,
                angle_widget.value,
                mesh_widget.value,
            )
            
            # Generate visualization
            fig = create_visualization(report)
            if fig:
                fig.show()
                
                # Display report summary
                print("\n" + "="*60)
                print("SUBSIDENCE CALCULATION REPORT")
                print("="*60)
                print(f"Panel Points: {', '.join([f'({p['easting']:.1f},{p['northing']:.1f})' for p in panel_points])}")
                print(f"\nResults:")
                print(f"  Max Subsidence (s_max): {report['calculations']['s_max']:.4f} m ({report['calculations']['s_max']*1000:.1f} mm)")
                print(f"  Influence Distance: {report['calculations']['influence_distance']:.2f} m")
                print(f"  Influenced Points: {len(report['points'])}")
                print(f"  Average Subsidence: {report['calculations']['average_subsidence']:.4f} m")
                print(f"  Max Point Subsidence: {report['calculations']['max_subsidence']:.4f} m")
                print("="*60)
            else:
                print("No points available for visualization")
                
        except Exception as e:
            print(f"Error during calculation: {e}")
            import traceback
            traceback.print_exc()

# Connect button click to handler
calc_button.on_click(on_calculate)

print("✓ Calculation handler connected")

In [ ]:
# Build the UI layout
point_box = widgets.VBox([widgets.HBox(widg) for widg in point_inputs])

param_box = widgets.VBox([
    params_label,
    thickness_widget,
    depth_widget,
    extraction_widget,
    subsidence_factor_widget,
    new_ratio_widget,
    angle_widget,
    mesh_widget,
])

control_panel = widgets.VBox([
    title_label,
    num_points_widget,
    point_box,
    param_box,
    calc_button,
])

main_ui = widgets.HBox([control_panel, output_area], layout=widgets.Layout(width='100%'))

display(main_ui)

# Trigger initial calculation
calc_button.click()

---

## Section 3: Flask Web Server for Remote Access

The Flask web server provides a full-featured web interface accessible at `http://localhost:5000`

### Starting the Flask Server

Run the command below to start the web server (it will run in the background):

```bash
cd /workspaces/Subsidence && python web_app.py
```

Then open your browser to: **http://localhost:5000**

In [ ]:
import subprocess
import time
import requests
from threading import Thread

def start_flask_server():
    """Start Flask server in background"""
    try:
        # Check if server is already running
        response = requests.get('http://localhost:5000', timeout=1)
        print("✓ Flask server is already running at http://localhost:5000")
        return
    except:
        pass
    
    # Start server in background
    def run_server():
        os.chdir('/workspaces/Subsidence')
        subprocess.Popen([sys.executable, 'web_app.py'])
    
    server_thread = Thread(target=run_server, daemon=True)
    server_thread.start()
    
    # Wait for server to start
    for i in range(10):
        try:
            response = requests.get('http://localhost:5000', timeout=1)
            print("✓ Flask server started successfully!")
            print("  Open your browser to: http://localhost:5000")
            return
        except:
            time.sleep(1)
    
    print("⚠ Server may not have started. Try running manually or check for errors.")

# Uncomment the line below to start the server
# start_flask_server()

---

## Flask Features

**The web interface includes:**
- Interactive parameter sliders for real-time adjustment
- Dynamic panel point editing
- Live Plotly visualization updates
- Subsidence calculation results display
- Responsive design for desktop and tablet browsers

**To access:**
1. Run the cell above to start the Flask server
2. Open http://localhost:5000 in your browser
3. Adjust parameters using the web interface
4. Click "Calculate & Visualize" to update the interactive map

---

## Section 4: Advanced Analysis & Export

In [ ]:
def export_results_to_html(report, filename='subsidence_report.html'):
    """Export calculation results to standalone HTML file"""
    fig = create_visualization(report)
    if fig:
        filepath = f'/workspaces/Subsidence/{filename}'
        fig.write_html(filepath, include_plotlyjs='cdn')
        print(f"✓ Results exported to: {filepath}")
        return filepath
    return None

def export_results_to_csv(report, filename='subsidence_data.csv'):
    """Export grid points to CSV"""
    points = report['points']
    df = pd.DataFrame(points)
    filepath = f'/workspaces/Subsidence/{filename}'
    df.to_csv(filepath, index=False)
    print(f"✓ Data exported to: {filepath}")
    print(f"  Records: {len(df)}")
    return filepath

# Example: Export current results (uncomment to use)
# export_results_to_html(report, 'subsidence_map_export.html')
# export_results_to_csv(report, 'subsidence_points_export.csv')

---

## Quick Reference

### Visualization Options:

**1. Jupyter Notebook (Current)**
- ✓ Interactive widgets for parameter adjustment
- ✓ Real-time Plotly visualization
- ✓ Inline calculation results
- Click the "Calculate & Plot" button above to update

**2. Flask Web Server**
- ✓ Full-featured web interface
- ✓ Professional UI with gradients and styling
- ✓ Responsive design for all devices
- Start: `python web_app.py`
- Access: http://localhost:5000

**3. Standalone HTML**
- ✓ Export interactive maps
- ✓ Share with collaborators
- ✓ No server required
- Use export functions above

### Subsidence Model

The calculator uses:
- **Formula**: s_max = h × e × a
- **Influence**: d = H × tan(α)
- **Decay**: Quadratic falloff from panel boundary
- **Input**: Polygon panel boundary points
- **Output**: Grid of subsidence values with visualization

### Tips

1. Use the notebook for quick iterations
2. Use the web server for sharing results
3. Export HTML for presentations
4. Adjust mesh spacing for resolution vs. performance trade-offs
5. All calculations account for panel geometry via polygon centroid